In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

import joblib
from pathlib import Path

current_feature_dir =Path.cwd()
parent_dir = current_feature_dir.parent.parent
cleaned_data_dir = parent_dir / "data" / "processed"
cleand_csv_file=cleaned_data_dir / "laptop_price_cleaned.csv"
computer_cleaned_csv_file=cleaned_data_dir / "computer_prices_all_cleaned.csv"

df =pd.read_csv(computer_cleaned_csv_file)

# Parsing the cpu_model and gpu_model

def parse_cpu(cpu_model):
    """
    Parse CPU model into:
        - cpu_series
        - cpu_generation
        - cpu_suffix
    """

    if pd.isna(cpu_model):
        return pd.Series(
            [np.nan, np.nan, np.nan],
            index=[
                "cpu_series",
                "cpu_generation",
                "cpu_suffix"
            ]
        )

    text = str(cpu_model).strip()

    cpu_series = np.nan
    cpu_generation = np.nan
    cpu_suffix = np.nan

    # -------------------------
    # AMD Ryzen
    # Example:
    # AMD Ryzen 7 7840HS
    # -------------------------

    amd = re.search(
        r"Ryzen\s+([3579])\s+(\d{4,5})([A-Za-z]*)",
        text,
        flags=re.IGNORECASE
    )

    if amd:

        cpu_series = f"Ryzen {amd.group(1)}"

        model_number = amd.group(2)

        cpu_generation = model_number[0]

        suffix = amd.group(3).upper()

        cpu_suffix = suffix if suffix else "Standard"

        return pd.Series(
            [
                cpu_series,
                cpu_generation,
                cpu_suffix
            ],
            index=[
                "cpu_series",
                "cpu_generation",
                "cpu_suffix"
            ]
        )

    # -------------------------
    # Intel Core
    # Example:
    # Intel i7-13620H
    # Intel i5-1135G7
    # -------------------------

    intel = re.search(
        r"(i[3579])[- ]?(\d{4,5})([A-Za-z0-9]*)",
        text,
        flags=re.IGNORECASE
    )

    if intel:

        cpu_series = intel.group(1).lower()

        model_number = intel.group(2)

        if len(model_number) == 5:
            cpu_generation = model_number[:2]
        else:
            cpu_generation = model_number[0]

        suffix = intel.group(3).upper()

        cpu_suffix = suffix if suffix else "Standard"

        return pd.Series(
            [
                cpu_series,
                cpu_generation,
                cpu_suffix
            ],
            index=[
                "cpu_series",
                "cpu_generation",
                "cpu_suffix"
            ]
        )
    apple = re.search(
        r"M([1234])\s*(Pro|Max|Ultra)?", text, flags=re.IGNORECASE
    )

    if apple:

        cpu_series = "M"

        cpu_generation = apple.group(1)

        suffix = apple.group(2)

        if suffix is None:
             suffix = "Standard"

        return pd.Series(
            
            
            [
                cpu_series,
                cpu_generation,
                suffix
            ],
            index=[
                "cpu_series",
                "cpu_generation",
                "cpu_suffix"
            ]
        )

    return pd.Series(
        [
        "Unknown",
        -1,
        "Unknown"
    ],
        index=[
            "cpu_series",
            "cpu_generation",
            "cpu_suffix"
        ]
    )
#Apply parser
cpu_features = df["cpu_model"].apply(parse_cpu)
df = pd.concat([df, cpu_features], axis=1)

df["cpu_generation"] = (
    pd.to_numeric(
        df["cpu_generation"],
        errors="coerce"
    )
    .fillna(-1)
    .astype(int)
    
)
unknown = df[df["cpu_series"] == "Unknown"]

print(unknown[["cpu_brand", "cpu_model"]].head(50))



# GPU_model parse
def parse_gpu(gpu_model):

    if pd.isna(gpu_model):
        return pd.Series(
            ["Unknown", -1, "Unknown"],
            index=[
                "gpu_family",
                "gpu_generation",
                "gpu_suffix"
            ]
        )

    text = str(gpu_model).strip()

    # ------------------------
    # NVIDIA RTX
    # Example:
    # Rtx 40 60
    # ------------------------

    rtx = re.search(
        r"RTX\s+(\d+)\s+(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if rtx:

        return pd.Series(
            [
                "RTX",
                f"RTX{rtx.group(1)}",
                rtx.group(2)
            ],
            index=[
                "gpu_family",
                "gpu_generation",
                "gpu_suffix"
            ]
        )

    # ------------------------
    # AMD RX
    # Example:
    # Rx 7000 60
    # ------------------------

    rx = re.search(
        r"RX\s+(\d+)\s+(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if rx:

        return pd.Series(
            [
                "RX",
                f"R{rx.group(1)}",
                rx.group(2)
            ],
            index=[
                "gpu_family",
                "gpu_generation",
                "gpu_suffix"
            ]
        )

    # ------------------------
    # Intel ARC
    # ------------------------

    arc = re.search(
        r"Arc\s+(A\d+)\s*(.*)",
        text,
        flags=re.IGNORECASE
    )

    if arc:

        suffix = arc.group(2).strip()

        if suffix == "":
            suffix = "Standard"

        return pd.Series(
            [
                "ARC",
                arc.group(1),
                suffix
            ],
            index=[
                "gpu_family",
                "gpu_generation",
                "gpu_suffix"
            ]
        )

    # ------------------------
    # Apple
    # ------------------------

    if "Apple" in text:

        return pd.Series(
            [
                "Apple",
                "Integrated",
                "Standard"
            ],
            index=[
                "gpu_family",
                "gpu_generation",
                "gpu_suffix"
            ]
        )

    return pd.Series(
        [
            "Unknown",
            -1,
            "Unknown"
        ],
        index=[
            "gpu_family",
            "gpu_generation",
            "gpu_suffix"
        ]
    )
#Applying;
gpu_features = df["gpu_model"].apply(parse_gpu)

df = pd.concat([df, gpu_features], axis=1)

 
# Parse resolution
def parse_resolution(resolution):
    if pd.isna(resolution):
        return pd.Series(
            [np.nan, np.nan, np.nan],
            index=["resolution_width", "resolution_height", "megapixels"]
        )

    text = str(resolution).strip()

    # ✅ Case‑insensitive match for x, X, or ×
    match = re.search(r"(\d+)\s*[xX×]\s*(\d+)", text, flags=re.IGNORECASE)

    if match:
        width = int(match.group(1))
        height = int(match.group(2))
        megapixels = round((width * height) / 1_000_000, 2)
        return pd.Series(
            [width, height, megapixels],
            index=["resolution_width", "resolution_height", "megapixels"]
        )

    # Fallback: try splitting on common separators
    for sep in ['x', 'X', '×']:
        if sep in text:
            parts = text.split(sep)
            if len(parts) == 2:
                try:
                    width = int(parts[0].strip())
                    height = int(parts[1].strip())
                    megapixels = round((width * height) / 1_000_000, 2)
                    return pd.Series(
                        [width, height, megapixels],
                        index=["resolution_width", "resolution_height", "megapixels"]
                    )
                except ValueError:
                    pass

    return pd.Series(
        [np.nan, np.nan, np.nan],
        index=["resolution_width", "resolution_height", "megapixels"]
    )

for col in ["resolution_width", "resolution_height", "megapixels"]:
    if col in df.columns:
        df = df.drop(columns=[col])

#Apply
resolution_features = df["resolution"].apply(parse_resolution)
df = pd.concat([df, resolution_features], axis=1)

# --- Save the fully processed dataset ---
final_csv_path = cleaned_data_dir / "computer_prices_final.csv"
df.to_csv(final_csv_path, index=False)
print(f"Saved final processed data to {final_csv_path}")


# Separate Features and Target
TARGET = "price"
# cols_to_drop=[
#     "model",
#     "cpu_model",
#     "resolution",
#     "gpu_model"
# ]

# df=df.drop(columns =cols_to_drop)
raw_or_parse_columns = [
    "model",
    "cpu_model", 
    "gpu_model",
    "resolution"
]
exclude_from_X = raw_or_parse_columns
X = df.drop(columns=[TARGET] + exclude_from_X)
print(X.columns.tolist())

y = df[TARGET]

# Identify Feature Types

numerical_features = X.select_dtypes(  include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
# change categorical_features to string
# Convert categorical features to string (ensures uniform dtype for OneHotEncoder)
for col in categorical_features:
    X[col] = X[col].astype(str)

print("Numerical Features")
print(numerical_features)

print()

print("Categorical Features")
print(categorical_features)

# Build the Preprocessing Pipeline
#numerical_transformer

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)
# categorical Transoformer
categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                drop="if_binary",
                min_frequency=10             
            )
        )
    ]
)
#Combine both pipelines

preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_transformer,
            numerical_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features
        )

    ]

)

# Train and spit part

X_train, X_test, y_train, y_test =train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Fit the processor

preprocessor.fit(X_train)

# Transform the data

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed=preprocessor.transform(X_test)
print(f"transformed x_train:{X_train_transformed.shape}")
print(f"transformed x_test :{X_test_transformed.shape}")

print("\nTraining Columns")
print("=" * 60)

for col in X.columns:
    print(col, X[col].dtype)

# Save the preprocessor
joblib.dump(preprocessor,"../../artifacts/computer_preprocessor.joblib" )

print(df[[
    "cpu_tier",
    "gpu_tier",
    "cpu_series",
    "gpu_family",
    "gpu_generation",
    "bluetooth",
    "cpu_generation"
]].head(10))

print(df[[
    "cpu_tier",
    "gpu_tier",
    "cpu_series",
    "gpu_family",
    "gpu_generation",
    "bluetooth"
]].head(10))

Empty DataFrame
Columns: [cpu_brand, cpu_model]
Index: []
Saved final processed data to d:\AI Data Science and MERN\Projects\Laptop-Price\backend\data\processed\computer_prices_final.csv
['device_type', 'brand', 'release_year', 'os', 'form_factor', 'cpu_brand', 'cpu_tier', 'cpu_cores', 'cpu_threads', 'cpu_base_ghz', 'cpu_boost_ghz', 'gpu_brand', 'gpu_tier', 'vram_gb', 'ram_gb', 'storage_type', 'storage_gb', 'storage_drive_count', 'display_type', 'display_size_in', 'refresh_hz', 'battery_wh', 'charger_watts', 'psu_watts', 'wifi', 'bluetooth', 'weight_kg', 'warranty_months', 'cpu_series', 'cpu_generation', 'cpu_suffix', 'gpu_family', 'gpu_generation', 'gpu_suffix', 'resolution_width', 'resolution_height', 'megapixels']
Numerical Features
['release_year', 'cpu_tier', 'cpu_cores', 'cpu_threads', 'cpu_base_ghz', 'cpu_boost_ghz', 'gpu_tier', 'vram_gb', 'ram_gb', 'storage_gb', 'storage_drive_count', 'display_size_in', 'refresh_hz', 'battery_wh', 'charger_watts', 'psu_watts', 'bluetooth', 'wei